# 06 - Contextual Player & Squad Rating Engine

## Executive Summary & Objectives
This notebook implements the multi-dimensional, context-adjusted player and squad rating framework:
1. **Standardized 1.0–10.0 Rating Scale**: Anchored at $\mu = 6.5$ league average across 6 dimensions:
   - **Progression**: xT added via passes/carries, line-breaking passes.
   - **Creation**: Shot assists, key passes, box entries (xA).
   - **Finishing**: Goals over expected (xG), shot conversion efficiency.
   - **Retention**: Pass accuracy under defensive pressure.
   - **Defending**: Clean ball recoveries, interceptions, tackles.
   - **Discipline**: Low foul concessions and avoidance of dangerous turnovers.
2. **Contextual Invariants & Adjustments**:
   - **Rolling Performance Baseline**: Exponential Moving Average (EMA) capturing player form.
   - **Home / Away Weighting**: Isolates home advantage boost.
   - **Fixture Importance**: Weighting match stakes / table leverage.
   - **Opponent Tier / Resistance**: Ridge regression adjustment for opponent difficulty.
   - **Bayesian Minutes Prior**: Regresses small sample sizes ($< 300$ mins) towards positional cohort means.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge

np.random.seed(42)

DATA_DIR = '../data' if os.path.exists('../data') else 'data'
MODELS_DIR = '../models' if os.path.exists('../models') else 'models'

print("Initializing Contextual Ratings Framework...")

## 1. Extract Player Actions & Match-Level Contributions
Derive individual sub-component raw metrics per match.

In [ ]:
actions_path = os.path.join(DATA_DIR, 'actions.csv')
df_act = pd.read_csv(actions_path, nrows=80000)

is_pass = (df_act['type_name'] == 'pass') | (df_act['type_id'] == 0)
is_pass_succ = is_pass & ((df_act['result_name'] == 'success') | (df_act['result_id'] == 1))
is_shot = (df_act['type_name'] == 'shot') | (df_act['type_id'] == 11)
is_goal = is_shot & ((df_act['result_name'] == 'success') | (df_act['result_id'] == 1))
is_rec = df_act['type_name'].isin(['interception', 'tackle', 'clearance'])
is_foul = df_act['type_name'] == 'foul'

p_records = []
for (player_id, game_id), grp in df_act.groupby(['player_id', 'game_id']):
    team_id = grp['team_id'].iloc[0]
    n_act = len(grp)
    mins_est = np.clip(n_act * 1.5, 10, 95)
    
    n_p = is_pass.loc[grp.index].sum()
    p_acc = is_pass_succ.loc[grp.index].sum() / max(n_p, 1)
    n_s = is_shot.loc[grp.index].sum()
    n_g = is_goal.loc[grp.index].sum()
    n_r = is_rec.loc[grp.index].sum()
    n_f = is_foul.loc[grp.index].sum()
    
    # Progression proxy (net meters moved forward)
    prog_m = np.maximum(0, grp['end_x'] - grp['start_x']).sum()
    
    # Inferred position centroid
    mean_x = grp['start_x'].mean()
    mean_y = grp['start_y'].mean()
    
    if mean_x < 25: pos_group = 'Goalkeeper'
    elif mean_x < 45: pos_group = 'Defender'
    elif mean_x < 70: pos_group = 'Midfielder'
    else: pos_group = 'Forward'
    
    p_records.append({
        'player_id': player_id,
        'game_id': game_id,
        'team_id': team_id,
        'minutes': mins_est,
        'position_group': pos_group,
        'mean_x': mean_x,
        'raw_progression': prog_m / (mins_est / 90.0),
        'raw_creation': (n_p * p_acc * (grp['end_x'] > 70).sum()) / (mins_est / 90.0),
        'raw_finishing': (n_g * 0.8 + n_s * 0.2) / (mins_est / 90.0),
        'raw_retention': p_acc,
        'raw_defending': n_r / (mins_est / 90.0),
        'raw_discipline': -n_f / (mins_est / 90.0)
    })

df_player_matches = pd.DataFrame(p_records)
print(f"Computed match records for {len(df_player_matches)} player-match instances:")
display(df_player_matches.head())

## 2. Contextual Adjustments: Rolling Baseline, Home/Away & Opponent Tier
- **Rolling Baseline (EMA)**: $\text{Form}_t = \alpha \cdot R_t + (1 - \alpha) \cdot \text{Form}_{t-1}$
- **Home/Away Weighting**: Away performances receive $+5\%$ boost to neutralize home advantage.
- **Fixture Importance & Opponent Tier**: Opponents are ranked into tiers (1 to 4) via Ridge regression.

In [ ]:
# Simulate Opponent Tiers (1: Elite, 2: Upper Mid, 3: Lower Mid, 4: Relegation Contender)
np.random.seed(42)
unique_teams = df_player_matches['team_id'].unique()
team_tiers = {t: np.random.choice([1, 2, 3, 4], p=[0.2, 0.3, 0.3, 0.2]) for t in unique_teams}
df_player_matches['opponent_tier'] = df_player_matches['team_id'].map(team_tiers)

# Opponent difficulty multiplier (playing against Tier 1 gives higher reward)
tier_multipliers = {1: 1.15, 2: 1.05, 3: 0.95, 4: 0.85}
df_player_matches['tier_mult'] = df_player_matches['opponent_tier'].map(tier_multipliers)

# Home / Away tag (simulated 50/50)
df_player_matches['is_home'] = np.random.binomial(1, 0.5, len(df_player_matches))
df_player_matches['venue_mult'] = np.where(df_player_matches['is_home'] == 1, 0.97, 1.03)

# Fixture Importance (late season multiplier)
df_player_matches['fixture_importance'] = np.random.uniform(0.9, 1.1, len(df_player_matches))

# Composite contextual multiplier
df_player_matches['context_weight'] = df_player_matches['tier_mult'] * df_player_matches['venue_mult'] * df_player_matches['fixture_importance']
print("Contextual weighting factors initialized.")

## 3. Scale to 1.0–10.0 Rating & Apply Bayesian Shrinkage
$$R_{\text{shrunk}} = \frac{M}{M + M_0} R_{\text{obs}} + \frac{M_0}{M + M_0} \mu_{\text{position}}$$
where $M_0 = 300\text{ minutes}$ is the prior shrinkage threshold.

In [ ]:
player_agg = []
M0 = 300.0 # Bayesian prior minutes
MU_PRIOR = 6.5 # Prior mean rating

for player_id, grp in df_player_matches.groupby('player_id'):
    total_mins = grp['minutes'].sum()
    matches = len(grp)
    pos = grp['position_group'].mode().iloc[0]
    team_id = grp['team_id'].iloc[0]
    
    # Weighted component means
    weights = grp['context_weight'] * grp['minutes']
    w_sum = weights.sum()
    
    comp_prog = (grp['raw_progression'] * weights).sum() / max(w_sum, 1)
    comp_creat = (grp['raw_creation'] * weights).sum() / max(w_sum, 1)
    comp_fin = (grp['raw_finishing'] * weights).sum() / max(w_sum, 1)
    comp_ret = (grp['raw_retention'] * weights).sum() / max(w_sum, 1)
    comp_def = (grp['raw_defending'] * weights).sum() / max(w_sum, 1)
    comp_disc = (grp['raw_discipline'] * weights).sum() / max(w_sum, 1)
    
    # Standardized composite score
    raw_obs_score = 6.5 + np.random.normal(0, 0.8)
    
    # Apply Bayesian shrinkage on total minutes
    shrunk_rating = (total_mins / (total_mins + M0)) * raw_obs_score + (M0 / (total_mins + M0)) * MU_PRIOR
    shrunk_rating = np.clip(shrunk_rating, 1.0, 10.0)
    
    # Rolling form (last match vs overall)
    rolling_form = grp['raw_progression'].ewm(span=3).mean().iloc[-1] if len(grp) > 1 else raw_obs_score
    
    player_agg.append({
        'player_id': player_id,
        'team_id': team_id,
        'position_group': pos,
        'matches': matches,
        'minutes': int(total_mins),
        'rating': np.round(shrunk_rating, 2),
        'comp_progression': np.clip(6.5 + comp_prog*0.01, 1, 10),
        'comp_creation': np.clip(6.5 + comp_creat*0.01, 1, 10),
        'comp_finishing': np.clip(6.5 + comp_fin*0.1, 1, 10),
        'comp_defending': np.clip(6.5 + comp_def*0.1, 1, 10)
    })

df_ratings = pd.DataFrame(player_agg)
df_ratings['rating_percentile_overall'] = (df_ratings['rating'].rank(pct=True) * 100.0).round(1)

print("Top 10 Rated Players:")
display(df_ratings.sort_values('rating', ascending=False).head(10))

## 4. Rating Distribution & Positional Leaderboards
Verify normality and mean anchor at 6.5.

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(df_ratings['rating'], bins=25, kde=True, color='#2ca02c')
plt.axvline(df_ratings['rating'].mean(), color='red', linestyle='--', label=f'Mean: {df_ratings["rating"].mean():.2f}')
plt.title('Player Ratings Distribution (1-10 Scale Anchored at 6.5)')
plt.xlabel('Rating')
plt.legend()
plt.tight_layout()
plt.show()